# Дообучение USER-bge-m3 на silver+gold датасете (Google Colab)

## Что нужно сделать перед запуском

### 1. Выбрать GPU-рантайм
**Runtime → Change runtime type → GPU** (T4 — минимум; если доступен A100/L4 — возьми его, тогда можно будет поднять батч).

### 2. Данные уже на Google Drive

На Drive должны лежать ровно те же два parquet-файла, которые использовались для обучения E5 — их больше создавать не надо:

```
Google Drive/
└── thesis/
    └── data/
        ├── silver/
        │   └── train_gold_silver.parquet     ← train (gold_train + silver_balanced)
        └── golden/
            └── golden_eval.parquet            ← eval (20% golden, seed=42)
```

Используя **тот же** `golden_eval.parquet`, что и для E5, мы получаем корректное попарное сравнение моделей на одном и том же holdout'е.

### 3. Куда попадёт результат

```
Google Drive/
└── thesis/
    └── models/
        └── bi-encoder-bge-m3-finetuned.tar.gz    ← готово к скачиванию
```

Чекпоинты тренера пишутся **сразу на Drive** в `thesis/training/bi-encoder-bge-m3/` — это медленнее, чем `/content/`, но переживает любой дисконнект Colab (лимит GPU, закрытие вкладки, истечение runtime). При повторном запуске тренер автоматически продолжит с последнего сохранённого чекпоинта.

### 4. Что делать локально после обучения

С Google Drive забираешь `thesis/models/bi-encoder-bge-m3-finetuned.tar.gz` и распаковываешь в `thesis/models/`:

```
cd thesis/models/
tar -xzf /path/to/bi-encoder-bge-m3-finetuned.tar.gz
```

Появится папка `thesis/models/bi-encoder-bge-m3-finetuned/`. Дальше она загружается как обычный `SentenceTransformer` из локального пути.

### 5. Отличия от E5-ноутбука (важные для обоснования в дипломе)

- **Модель**: `deepvk/USER-bge-m3` (~568M параметров, 1024d, XLM-RoBERTa-large класс).
- **Префиксы не используются**: USER-bge-m3 — не E5, в него на вход идут чистые тексты без `query: ` / `passage: `.
- **Батч уменьшен до 16** + `gradient_accumulation_steps=2` → эффективный батч всё равно 32, как у E5, а VRAM хватает на T4.
- **Learning rate чуть ниже** (1e-5 против 2e-5 у E5) — стандартная практика для более крупных моделей.
- **`max_seq_length = 256`** — совпадает с тем, что используется в пайплайне индексации (`thesis/db/create-50k-colab.ipynb`), плюс экономит память.
- **Всё остальное идентично**: CoSENTLoss, 4 эпохи, тот же seed=42, тот же `gold_eval` как holdout, те же `train_gold_silver.parquet`. Это гарантирует честность сравнения RoSBERTa / E5 / bge-m3.


## 1. Установка зависимостей и монтирование Google Drive


In [ ]:
# Установка зависимостей и монтирование Google Drive
#
# ВАЖНО: флаг --upgrade-strategy only-if-needed запрещает pip'у трогать torch.
# Иначе pip подтянет свежий CPU-only torch и GPU работать перестанет.
!pip install -q --upgrade-strategy only-if-needed sentence-transformers datasets pyarrow
!pip uninstall -y triton 2>/dev/null; true

from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Жёсткая проверка: если CUDA недоступна — дальше нет смысла
assert torch.cuda.is_available(), (
    "CUDA недоступен. Причины:\n"
    "  1) Не выбран GPU-рантайм: Runtime → Change runtime type → GPU.\n"
    "  2) pip установил CPU-only torch поверх родного. Runtime → Disconnect "
    "and delete runtime, затем запусти ноутбук заново."
)
print(f"Устройство: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 2. Пути и проверка файлов


In [ ]:
import os
import shutil

# Базовый путь на Google Drive
DRIVE_BASE = "/content/drive/MyDrive/thesis"

# Входные parquet-файлы (те же, что и у E5 — один и тот же сплит)
train_parquet_path = os.path.join(DRIVE_BASE, "data/silver/train_gold_silver.parquet")
eval_parquet_path  = os.path.join(DRIVE_BASE, "data/golden/golden_eval.parquet")

# Папка для результата на Drive (сюда положим архив модели)
drive_models_dir = os.path.join(DRIVE_BASE, "models")
os.makedirs(drive_models_dir, exist_ok=True)

# ВАЖНО: чекпоинты тренера пишем СРАЗУ на Drive, а не в /content/.
# Иначе при дисконнекте Colab (лимиты, таймаут, истечение GPU-квоты)
# вся тренировка превращается в тыкву — /content/ — эфемерный диск.
# Стоимость: каждый save_steps добавляет ~30-60с из-за медленного Drive I/O,
# при ~9 сохранениях за обучение это +5 минут к общему времени. Терпимо.
TRAINER_OUTPUT_DIR     = os.path.join(DRIVE_BASE, "training/bi-encoder-bge-m3")  # чекпоинты на Drive
LOCAL_FINAL_MODEL_DIR  = "/content/bi-encoder-bge-m3-finetuned"                  # финальная модель — локально, потом архивируем на Drive
os.makedirs(TRAINER_OUTPUT_DIR, exist_ok=True)

# Проверка входных файлов
assert os.path.exists(train_parquet_path), f"Файл не найден: {train_parquet_path}"
assert os.path.exists(eval_parquet_path),  f"Файл не найден: {eval_parquet_path}"

print(f"Train: {train_parquet_path}")
print(f"  размер: {os.path.getsize(train_parquet_path) / 1e6:.1f} MB")
print(f"Eval:  {eval_parquet_path}")
print(f"  размер: {os.path.getsize(eval_parquet_path) / 1e6:.1f} MB")
print(f"Папка для результата: {drive_models_dir}")


## 3. Параметры обучения


In [ ]:
# ==================== ПАРАМЕТРЫ ====================
BASE_MODEL = "deepvk/USER-bge-m3"   # 1024d, ~568M параметров, XLM-RoBERTa-large class

# Память на T4 (15 GB) для bge-m3:
#   - fp16 + gradient_checkpointing обязательны
#   - BATCH_SIZE=16 на T4 влезает с запасом (~10-12 GB peak)
#   - эффективный батч = 16 * 2 = 32, как у E5 — чтобы шаги оптимизации были сопоставимы
BATCH_SIZE            = 16
GRAD_ACCUM_STEPS      = 2
LEARNING_RATE         = 1e-5      # чуть ниже, чем у E5 (2e-5) — модель больше, ей хватает меньшего шага
# Изначально стояло 4 эпохи (как у E5). По данным первого прерванного запуска
# (см. notes/data_isolation.md) Spearman вышел на плато к шагу ~2500-3500:
#   500: 0.475  →  1500: 0.539  →  2500: 0.571  →  3000: 0.567  →  3500: 0.577
# Поэтому 3 эпохи — достаточно, и заодно сокращаем риск упереться в лимит GPU.
NUM_EPOCHS            = 3
WARMUP_RATIO          = 0.1
EVAL_STEPS            = 500
SAVE_STEPS            = 500
SAVE_TOTAL_LIMIT      = 3
LOGGING_STEPS         = 50
SEED                  = 42

# max_seq_length: совпадает с тем, что будет использовано в инференсе (create-50k-colab).
# У bge-m3 по умолчанию 8192 — без явного ограничения съест всю VRAM на длинных постах.
MAX_SEQ_LEN           = 256

# USER-bge-m3 не использует query/passage-префиксы — в отличие от E5.
USE_PREFIX = False


## 4. Загрузка и препроцессинг датасетов


In [ ]:
from datasets import load_dataset

train_dataset = load_dataset("parquet", data_files=train_parquet_path, split="train")
eval_dataset  = load_dataset("parquet", data_files=eval_parquet_path,  split="train")

print(f"Train: {len(train_dataset):,} пар, колонки: {train_dataset.column_names}")
print(f"Eval:  {len(eval_dataset):,} пар, колонки: {eval_dataset.column_names}")
print()
print("Пример train:")
print(train_dataset[0])


In [ ]:
# Для bge-m3 префиксы не добавляем (в отличие от E5).
# Колонка с запросом называется product_desc (fallback: description).

eval_col1 = "product_desc" if "product_desc" in train_dataset.column_names else "description"
print(f"Колонка запроса: {eval_col1}")

if USE_PREFIX:
    # Ветка на всякий случай — если вдруг понадобится экспериментировать.
    def add_prefixes(example):
        return {
            eval_col1:   "query: "   + example[eval_col1],
            "post_text": "passage: " + example["post_text"],
        }
    train_dataset = train_dataset.map(add_prefixes, desc="prefix train")
    eval_dataset  = eval_dataset.map(add_prefixes,  desc="prefix eval")
    print("Префиксы ДОБАВЛЕНЫ (USE_PREFIX=True)")
else:
    print("Префиксы НЕ добавлены — USER-bge-m3 использует чистые тексты.")


## 5. Модель и loss


In [ ]:
# Совместимость sentence-transformers с новыми transformers
import transformers
from transformers.modeling_utils import PreTrainedModel
transformers.PreTrainedModel = PreTrainedModel

from sentence_transformers import SentenceTransformer
from sentence_transformers.losses import CoSENTLoss

model = SentenceTransformer(BASE_MODEL, device="cuda")

# Ограничиваем длину входа — у bge-m3 по умолчанию 8192, нам столько не нужно.
model.max_seq_length = MAX_SEQ_LEN

print(f"Модель: {BASE_MODEL}")
print(f"Max seq length: {model.max_seq_length}")
print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")
print(f"Similarity: {model.similarity_fn_name}")

# Счётчик параметров — полезно зафиксировать в дипломе
n_params = sum(p.numel() for p in model.parameters())
print(f"Параметров: {n_params/1e6:.1f}M")

loss = CoSENTLoss(model)
print(f"\nLoss: CoSENTLoss")


## 6. Baseline — Spearman на golden до обучения


In [ ]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction

evaluator = EmbeddingSimilarityEvaluator(
    sentences1=eval_dataset[eval_col1],
    sentences2=eval_dataset["post_text"],
    scores=eval_dataset["score"],
    batch_size=BATCH_SIZE,
    main_similarity=SimilarityFunction.COSINE,
    name="golden",
    show_progress_bar=True,
)

print("Baseline (до обучения):")
baseline_results = evaluator(model)
baseline_spearman = baseline_results["golden_spearman_cosine"]
print(f"  Spearman (cosine): {baseline_spearman:.4f}")
print(f"  Pearson  (cosine): {baseline_results['golden_pearson_cosine']:.4f}")


## 7. Training arguments и запуск


In [ ]:
from sentence_transformers import SentenceTransformerTrainingArguments

args = SentenceTransformerTrainingArguments(
    output_dir=TRAINER_OUTPUT_DIR,

    # Длительность
    num_train_epochs=NUM_EPOCHS,

    # Батч + grad accumulation (эффективный батч = 16 * 2 = 32, как у E5)
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    # Оптимизатор
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,

    # Память: для bge-m3 на T4 это критично
    gradient_checkpointing=True,
    dataloader_num_workers=2,
    fp16=True,

    # Eval
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model="eval_golden_spearman_cosine",
    greater_is_better=True,

    # Checkpoints
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,

    # Логирование
    logging_steps=LOGGING_STEPS,

    # Воспроизводимость
    seed=SEED,
    data_seed=SEED,

    dataloader_drop_last=True,
)

# С учётом gradient_accumulation_steps шаги считаются по эффективному батчу
effective_bs = BATCH_SIZE * GRAD_ACCUM_STEPS
total_steps = (len(train_dataset) // effective_bs) * NUM_EPOCHS
print(f"Эффективный батч: {effective_bs}")
print(f"Ожидаемое кол-во шагов: ~{total_steps:,}")
print(f"Eval каждые {EVAL_STEPS} шагов = ~{max(total_steps // EVAL_STEPS, 1)} раз за обучение")


In [ ]:
from sentence_transformers import SentenceTransformerTrainer

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    evaluator=evaluator,
)

# Автоматический resume: если в TRAINER_OUTPUT_DIR есть checkpoint — продолжить с него
has_checkpoint = (
    os.path.isdir(TRAINER_OUTPUT_DIR)
    and any(d.startswith("checkpoint-") for d in os.listdir(TRAINER_OUTPUT_DIR))
)

if has_checkpoint:
    print("Найден checkpoint, продолжаем обучение...")
else:
    print("Checkpoint не найден, начинаем с нуля...")

trainer.train(resume_from_checkpoint=has_checkpoint if has_checkpoint else None)


## 8. Сохранение финальной модели


In [ ]:
# Локально
model.save_pretrained(LOCAL_FINAL_MODEL_DIR)
print(f"Модель сохранена локально: {LOCAL_FINAL_MODEL_DIR}")
print(f"  размер: {sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(LOCAL_FINAL_MODEL_DIR) for f in fs) / 1e6:.1f} MB")


## 9. Финальная оценка


In [ ]:
print("Финальная оценка на golden dataset:")
final_results = evaluator(model)
final_spearman = final_results["golden_spearman_cosine"]
final_pearson = final_results["golden_pearson_cosine"]

print(f"\n{'='*50}")
print(f"{'РЕЗУЛЬТАТЫ (USER-bge-m3)':^50}")
print(f"{'='*50}")
print(f"{'Метрика':<25} {'До':>10} {'После':>10} {'Delta':>10}")
print(f"{'-'*50}")
print(f"{'Spearman (cosine)':<25} {baseline_spearman:>10.4f} {final_spearman:>10.4f} {final_spearman - baseline_spearman:>+10.4f}")
print(f"{'Pearson (cosine)':<25} {baseline_results['golden_pearson_cosine']:>10.4f} {final_pearson:>10.4f} {final_pearson - baseline_results['golden_pearson_cosine']:>+10.4f}")
print(f"{'='*50}")


## 10. Архивирование и копирование на Google Drive


In [ ]:
# Архивируем локальную папку с моделью и копируем .tar.gz на Drive
ARCHIVE_NAME = "bi-encoder-bge-m3-finetuned.tar.gz"
local_archive = f"/content/{ARCHIVE_NAME}"

assert os.path.exists(LOCAL_FINAL_MODEL_DIR), f"Финальная модель не найдена: {LOCAL_FINAL_MODEL_DIR}"

# tar без лишней обёртки: после распаковки получится сразу bi-encoder-bge-m3-finetuned/
parent_dir = os.path.dirname(LOCAL_FINAL_MODEL_DIR)
base_name  = os.path.basename(LOCAL_FINAL_MODEL_DIR)

!tar -czf {local_archive} -C {parent_dir} {base_name}

archive_size_mb = os.path.getsize(local_archive) / 1e6
print(f"Архив создан: {local_archive} ({archive_size_mb:.1f} MB)")

# Копируем на Google Drive
drive_archive = os.path.join(drive_models_dir, ARCHIVE_NAME)
shutil.copy(local_archive, drive_archive)

assert os.path.exists(drive_archive), "Не удалось скопировать архив на Drive"
print(f"✓ Архив сохранён на Drive: {drive_archive}")
print(f"  Размер: {os.path.getsize(drive_archive) / 1e6:.1f} MB")


## Что делать локально после скачивания

С Google Drive забираешь `thesis/models/bi-encoder-bge-m3-finetuned.tar.gz` и распаковываешь в `thesis/models/`:

```bash
cd thesis/models/
tar -xzf /path/to/bi-encoder-bge-m3-finetuned.tar.gz
```

Появится `thesis/models/bi-encoder-bge-m3-finetuned/`. Дальше её можно использовать как обычный SentenceTransformer:

```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("models/bi-encoder-bge-m3-finetuned")
model.max_seq_length = 256  # совпадает с тем, что использовалось при обучении и индексации
```

**Важно:** в отличие от E5, для USER-bge-m3 никаких префиксов при инференсе не нужно — тексты подаются в чистом виде.

После этого:
1. В `thesis/db/create-50k-colab.ipynb` раскомментируй пресет `bge-m3-fine-tuned` (там уже прописан путь к этому архиву на Drive — ноутбук распакует его автоматически) и построй LanceDB-таблицу `bge-m3-fine-tuned-50k`.
2. Добавь бенчмарк-ноутбук в `thesis/benchmark/bi-encoder/bge-m3/` по аналогии с `rosberta/` и сравни base vs fine-tuned.
3. Добавь bge-m3 fine-tuned в `thesis/benchmark/pipeline/pipeline-50k.ipynb` для честного попарного сравнения с RoSBERTa и E5 — все три будут оценены на одном и том же `gold_eval` (seed=42).
